In [1]:
import os
import glob
import pandas as pd

def inspect_biomass_folder(file_path):
    """
    Dynamically locates the table header in raw or preprocessed files, loads the CSV,
    and outputs shape, column structure, and baseline statistics.
    """
    header_row_index = 0
    
    # Locate the telemetry table header row (works for both raw Agilent and _cleaned files)
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            # We look for standard raw headers or preprocessed headers like 'timestamp' / 'scan_num'
            if any(key in line.lower() for key in ['scan num', '101 (', 'scan swee', 'timestamp', 'scan_num']):
                header_row_index = idx
                break
                
    # Load dataset from the detected header row
    df = pd.read_csv(file_path, skiprows=header_row_index)
    
    # Clean column whitespace and drop completely empty rows or columns
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    return df

# ==========================================
# EXECUTION ON BIOMASS FOLDER
# ==========================================
biomass_path = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Biomass"

# We strictly match only .csv files, automatically ignoring '.keep' and other non-data files
csv_files = glob.glob(os.path.join(biomass_path, "*.csv"))

print(f"Found {len(csv_files)} valid CSV files in Biomass folder (ignoring .keep). Running inspection...\n")

for i, file_path in enumerate(csv_files, 1):
    file_name = os.path.basename(file_path)
    
    try:
        df = inspect_biomass_folder(file_path)
        
        print(f"File {i}: {file_name}")
        print(f"   Shape: {df.shape[0]} rows by {df.shape[1]} columns")
        print(f"   Columns: {list(df.columns)}")
        print(f"   Missing Values: {df.isnull().sum().sum()} total nulls")
        
        # Select numeric columns for basic baseline stats
        numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
        if len(numeric_cols) > 0:
            print("\n   Baseline Data Summary (First 5 Numeric Columns):")
            print(df[numeric_cols[:5]].describe().loc[['mean', 'min', 'max', 'std']])
        
        print("\n" + "="*65 + "\n")
        
    except Exception as e:
        print(f"Could not read {file_name}: {e}\n")

Found 6 valid CSV files in Biomass folder (ignoring .keep). Running inspection...

File 1: 1781367912_gasifier 50slpm 0.csv
   Shape: 796 rows by 7 columns
   Columns: ['Scan Sweep Time (Sec)', 'Scan Number', '101 (°C)', '102 (°C)', '103 (°C)', '104 (°C)', '105 (°C)']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 5 Numeric Columns):
      Scan Number     101 (°C)     102 (°C)    103 (°C)    104 (°C)
mean   398.500000   425.204070   495.063349  522.364208  520.175080
min      1.000000    27.770788    27.511129   30.257773   30.520025
max    796.000000  1171.626450  1107.979790  989.705587  889.174997
std    229.929699   427.698895   432.131571  359.681821  254.000610


File 2: 20260613T100035_Gasifier 1 0_cleaned.csv
   Shape: 714 rows by 9 columns
   Columns: ['timestamp', 'scan_number', 'ch_00', 'ch_01', 'ch_02', 'ch_03', 'ch_04', 'system_file', 'system_id']
   Missing Values: 0 total nulls

   Baseline Data Summary (First 5 Numeric Columns):
      scan_number     

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================================================
# 1. ADAPTIVE LOADER & OVERLOAD SCRUBBER
# =========================================================
def load_biomass_file(file_path):
    """
    Loads raw Agilent or preprocessed Biomass files, standardizes column names
    to 5 thermal zones, and scrubs overload error codes (+9.9E+37).
    """
    file_name = os.path.basename(file_path)
    
    if "cleaned" in file_name.lower():
        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()
        time_col = 'timestamp' if 'timestamp' in df.columns else df.index
        # Extract sensor feature columns
        sensor_cols = [col for col in df.columns if 'ch_' in col.lower()]
    else:
        # Scan for raw Agilent header row
        header_idx = 0
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            for idx, line in enumerate(f):
                if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee', '101Time']):
                    header_idx = idx
                    break
        df = pd.read_csv(file_path, skiprows=header_idx)
        df.columns = df.columns.str.strip()
        df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
        
        time_candidates = [col for col in df.columns if 'time' in col.lower() or 'swee' in col.lower()]
        time_col = time_candidates[0] if time_candidates else df.index
        sensor_cols = [col for col in df.columns if '°c' in col.lower()]

    # Standardize Timestamp column
    df['Timestamp_Clean'] = df[time_col] if isinstance(time_col, str) else df.index
    
    # SCRUB OVERLOAD ERRORS (+9.9E+37) & Standardize to float64
    clean_sensor_cols = []
    for idx, col in enumerate(sensor_cols[:5]):  # Focus on primary 5 zones
        clean_name = f"Zone_{idx} ({col})"
        df[clean_name] = pd.to_numeric(df[col], errors='coerce')
        # Replace extreme hardware overload constants with NaN
        df[clean_name] = df[clean_name].apply(lambda x: np.nan if (abs(x) > 10000 or x == np.inf) else x)
        clean_sensor_cols.append(clean_name)
            
    return df, clean_sensor_cols

# =========================================================
# 2. STATISTICAL ENGINE
# =========================================================
def compute_biomass_statistics(df, sensor_cols, file_name):
    """
    Computes univariate statistics for the 5 thermal reactor zones.
    """
    sensor_df = df[sensor_cols]
    
    print(f"================================================================")
    print(f" STATISTICAL SUMMARY: {file_name}")
    print(f"================================================================")
    
    stats_df = pd.DataFrame({
        'Mean': sensor_df.mean(),
        'Std Dev': sensor_df.std(),
        'Min': sensor_df.min(),
        '25% (Q1)': sensor_df.quantile(0.25),
        '50% (Median)': sensor_df.median(),
        '75% (Q3)': sensor_df.quantile(0.75),
        'Max': sensor_df.max(),
        'IQR': sensor_df.quantile(0.75) - sensor_df.quantile(0.25),
        'Skewness': sensor_df.skew(),
        'Kurtosis': sensor_df.kurtosis(),
        'Overloads Scrubbed (NaNs)': sensor_df.isnull().sum()
    })
    
    print(stats_df.round(2))
    print("================================================================\n")

# =========================================================
# 3. PER-FILE INTERACTIVE PLOTLY VISUALIZATIONS
# =========================================================
def plot_whole_dataset_bio(df, sensor_cols, file_name):
    """
    Renders the combined 5-zone time-series chart for a single file.
    """
    fig = px.line(
        df,
        x='Timestamp_Clean',
        y=sensor_cols,
        title=f"Whole-Dataset Gasifier Telemetry (5 Zones) — {file_name}",
        labels={'value': 'Temperature (°C)', 'variable': 'Reactor Zone', 'Timestamp_Clean': 'Timestamp / Step'}
    )
    fig.update_layout(
        hovermode='x unified',
        xaxis=dict(rangeslider=dict(visible=True), type='category'),
        height=450,
        width=1200
    )
    fig.show()

def plot_individual_features_bio(df, sensor_cols, file_name):
    """
    Renders a 5-row subplot grid pairing each zone's time-series trend 
    with its statistical box plot.
    """
    num_sensors = len(sensor_cols)
    fig = make_subplots(
        rows=num_sensors,
        cols=2,
        column_widths=[0.75, 0.25],
        subplot_titles=[item for sublist in [[f"{col} - Trend", f"{col} - Outlier Box Plot"] for col in sensor_cols] for item in sublist],
        horizontal_spacing=0.08,
        vertical_spacing=0.06
    )
    
    colors = px.colors.qualitative.Safe
    
    for idx, col in enumerate(sensor_cols):
        row_num = idx + 1
        color = colors[idx % len(colors)]
        
        # Left Column: Time-Series Trend
        fig.add_trace(
            go.Scatter(
                x=df['Timestamp_Clean'],
                y=df[col],
                mode='lines',
                name=col,
                line=dict(color=color, width=1.5),
                showlegend=False
            ),
            row=row_num, col=1
        )
        
        # Right Column: Box Plot
        fig.add_trace(
            go.Box(
                y=df[col].dropna(),
                name=col,
                marker_color=color,
                boxpoints='outliers',
                showlegend=False
            ),
            row=row_num, col=2
        )
        
        fig.update_yaxes(title_text="Temp (°C)", row=row_num, col=1)
        
    fig.update_layout(
        title_text=f"Individual Zone Inspection (5 Zones) — {file_name}",
        height=240 * num_sensors,
        width=1250,
        showlegend=False
    )
    fig.show()

# =========================================================
# 4. GLOBAL 6-CSV COMPARISON PLOT
# =========================================================
def plot_all_6_comparison(comparison_data):
    """
    Overlays the Primary Combustion Zone (Zone_0 / Channel 101 / ch_00) from all
    6 CSV files onto a single interactive Plotly figure for cross-experiment comparison.
    """
    fig = go.Figure()
    
    colors = px.colors.qualitative.Plotly
    
    for idx, (file_name, series) in enumerate(comparison_data.items()):
        color = colors[idx % len(colors)]
        
        fig.add_trace(
            go.Scatter(
                y=series.values,  # Plotted against scan index for uniform alignment
                mode='lines',
                name=file_name,
                line=dict(color=color, width=2.0),
                hovertemplate=f"<b>{file_name}</b><br>Scan Index: %{{x}}<br>Temp: %{{y:.2f}} °C<extra></extra>"
            )
        )
        
    fig.update_layout(
        title="<b>Cross-Experiment Comparison: Primary Combustion Zone across All 6 Runs</b>",
        xaxis_title="Relative Scan Step (Normalized Time Axis)",
        yaxis_title="Temperature (°C)",
        hovermode="x unified",
        legend=dict(title="Experiment / File Name", orientation="v", yanchor="top", y=1, xanchor="left", x=1.02),
        height=600,
        width=1300,
        xaxis=dict(rangeslider=dict(visible=True))
    )
    fig.show()

# =========================================================
# 5. EXECUTION LOOP Across All 6 Biomass CSVs
# =========================================================
biomass_dir = r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\Biomass"
csv_files = sorted(glob.glob(os.path.join(biomass_dir, "*.csv")))

print(f"Found {len(csv_files)} valid CSV files in Biomass folder. Processing each separately...\n")

# Dictionary to store Primary Zone series for the final 6-CSV overlay
primary_zone_comparison = {}

for i, sample_file in enumerate(csv_files, 1):
    file_name = os.path.basename(sample_file)
    print(f"\n======== PROCESSING FILE {i}/{len(csv_files)}: {file_name} ========")
    
    try:
        # 1. Load and scrub overload errors
        df_bio, sensors = load_biomass_file(sample_file)
        
        # 2. Compute statistics
        compute_biomass_statistics(df_bio, sensors, file_name)
        
        # 3. Generate Whole-Dataset Time-Series Plot
        plot_whole_dataset_bio(df_bio, sensors, file_name)
        
        # 4. Generate Individual Feature Box Plot & Trend Subplots
        plot_individual_features_bio(df_bio, sensors, file_name)
        
        # 5. Store the Primary Combustion Zone (Zone 0) for final comparison
        if len(sensors) > 0:
            primary_zone_comparison[file_name] = df_bio[sensors[0]].dropna()
            
    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")

# =========================================================
# 6. RENDER FINAL 6-CSV COMPARISON PLOT
# =========================================================
if primary_zone_comparison:
    print("\n======== GENERATING FINAL 6-CSV COMPARISON DASHBOARD ========")
    plot_all_6_comparison(primary_zone_comparison)

Found 6 valid CSV files in Biomass folder. Processing each separately...


======== PROCESSING FILE 1/6: 1781367912_gasifier 50slpm 0.csv ========
 STATISTICAL SUMMARY: 1781367912_gasifier 50slpm 0.csv
                     Mean  Std Dev    Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (101 (°C))  425.20   427.70  27.77     31.36        166.50    841.05   
Zone_1 (102 (°C))  495.06   432.13  27.51     33.51        760.73    894.49   
Zone_2 (103 (°C))  522.36   359.68  30.26     44.77        724.07    798.97   
Zone_3 (104 (°C))  520.18   254.00  30.52    276.79        605.36    667.88   
Zone_4 (105 (°C))  514.39   169.45  30.51    488.22        526.91    628.78   

                       Max     IQR  Skewness  Kurtosis  \
Zone_0 (101 (°C))  1171.63  809.68      0.37     -1.60   
Zone_1 (102 (°C))  1107.98  860.98     -0.04     -1.88   
Zone_2 (103 (°C))   989.71  754.20     -0.48     -1.63   
Zone_3 (104 (°C))   889.17  391.09     -0.74     -0.80   
Zone_4 (105 (°C))   803.74  140.5


======== PROCESSING FILE 2/6: 20260613T100035_Gasifier 1 0_cleaned.csv ========
 STATISTICAL SUMMARY: 20260613T100035_Gasifier 1 0_cleaned.csv
                  Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (ch_00)  179.56   317.39   29.81     31.16         32.09     61.89   
Zone_1 (ch_01)  242.62   298.47   30.56     31.79         43.18    524.43   
Zone_2 (ch_02)  263.74   207.57 -193.25     91.61        129.26    517.58   
Zone_3 (ch_03)  309.33   107.89   36.19    202.90        320.70    404.50   
Zone_4 (ch_04)  295.19    89.55   44.47    207.39        311.38    368.79   

                    Max     IQR  Skewness  Kurtosis  Overloads Scrubbed (NaNs)  
Zone_0 (ch_00)  1036.72   30.73      1.88      1.68                          0  
Zone_1 (ch_01)   925.40  492.64      1.01     -0.72                          0  
Zone_2 (ch_02)   575.73  425.97      0.38     -1.61                         35  
Zone_3 (ch_03)   578.56  201.60     -0.18     -0.96                  


======== PROCESSING FILE 3/6: 20260613T100255_gasifier 30lpm 0.csv ========
 STATISTICAL SUMMARY: 20260613T100255_gasifier 30lpm 0.csv
                     Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (101 (°C))  674.90   397.12   29.56    111.25        866.66    956.58   
Zone_1 (102 (°C))  649.65   211.83   51.90    678.94        696.24    719.22   
Zone_2 (103 (°C))  694.08    54.19  595.12    656.30        685.61    721.52   
Zone_3 (104 (°C))  592.00    26.24  504.24    576.05        590.91    608.19   
Zone_4 (105 (°C))  537.90    29.42  473.84    517.73        532.91    553.60   

                       Max     IQR  Skewness  Kurtosis  \
Zone_0 (101 (°C))  1084.43  845.33     -0.81     -1.11   
Zone_1 (102 (°C))   902.31   40.28     -2.08      3.24   
Zone_2 (103 (°C))   888.70   65.21      0.83      0.65   
Zone_3 (104 (°C))   754.56   32.15      0.96      5.86   
Zone_4 (105 (°C))   735.69   35.87      1.53      5.60   

                   Overloads Scrub


======== PROCESSING FILE 4/6: 20260614T040236_GASIFIER 2 0.csv ========
 STATISTICAL SUMMARY: 20260614T040236_GASIFIER 2 0.csv
                     Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (101 (°C))  998.57    76.33  861.30   1009.47       1017.76   1053.71   
Zone_1 (102 (°C))  904.33    67.79  841.41    848.01        886.39    939.35   
Zone_2 (103 (°C))  668.77    65.62  580.86    612.39        669.43    699.89   
Zone_3 (104 (°C))  549.02    72.20  461.62    479.01        552.94    585.44   
Zone_4 (105 (°C))  486.95    64.91  409.34    426.97        479.01    533.47   

                       Max     IQR  Skewness  Kurtosis  \
Zone_0 (101 (°C))  1080.16   44.24     -0.97     -0.53   
Zone_1 (102 (°C))  1023.50   91.34      0.82     -0.94   
Zone_2 (103 (°C))   821.19   87.50      0.55     -0.44   
Zone_3 (104 (°C))   694.10  106.44      0.56     -0.83   
Zone_4 (105 (°C))   598.05  106.50      0.34     -1.25   

                   Overloads Scrubbed (NaN


======== PROCESSING FILE 5/6: 20260614T040316_25112025_cleaned.csv ========
 STATISTICAL SUMMARY: 20260614T040316_25112025_cleaned.csv
                  Mean  Std Dev    Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (ch_00)  420.73   274.32 -82.22     34.79        509.31    598.34   
Zone_1 (ch_01)  332.31   258.77  32.22     81.66        284.79    585.03   
Zone_2 (ch_02)  233.16   160.74  31.54     78.50        203.54    411.48   
Zone_3 (ch_03)  175.91   115.83  31.82     71.19        191.21    289.31   
Zone_4 (ch_04)  189.82   130.29  32.18     65.62        206.59    315.66   

                   Max     IQR  Skewness  Kurtosis  Overloads Scrubbed (NaNs)  
Zone_0 (ch_00)  892.83  563.56     -0.27     -1.17                          0  
Zone_1 (ch_01)  748.11  503.37      0.03     -1.84                          0  
Zone_2 (ch_02)  463.28  332.98      0.11     -1.58                          0  
Zone_3 (ch_03)  366.79  218.13     -0.04     -1.76                          0  
Zone_4 


======== PROCESSING FILE 6/6: 20260614T074620_gasifier 30lpm 0_cleaned.csv ========
 STATISTICAL SUMMARY: 20260614T074620_gasifier 30lpm 0_cleaned.csv
                  Mean  Std Dev     Min  25% (Q1)  50% (Median)  75% (Q3)  \
Zone_0 (ch_00)  674.90   397.12   29.56    111.25        866.66    956.58   
Zone_1 (ch_01)  649.65   211.83   51.90    678.94        696.24    719.22   
Zone_2 (ch_02)  694.08    54.19  595.12    656.30        685.61    721.52   
Zone_3 (ch_03)  592.00    26.24  504.24    576.05        590.91    608.19   
Zone_4 (ch_04)  537.90    29.42  473.84    517.73        532.91    553.60   

                    Max     IQR  Skewness  Kurtosis  Overloads Scrubbed (NaNs)  
Zone_0 (ch_00)  1084.43  845.33     -0.81     -1.11                          0  
Zone_1 (ch_01)   902.31   40.28     -2.08      3.24                          0  
Zone_2 (ch_02)   888.70   65.21      0.83      0.65                          0  
Zone_3 (ch_03)   754.56   32.15      0.96      5.86          


======== GENERATING FINAL 6-CSV COMPARISON DASHBOARD ========
